In [ ]:
import pandas as pd
import os
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import glob
import plotnine as gg
import scipy.stats as stats
from tqdm import tqdm
import json
from essential.ode import ODEstimator
from essential.utils import load_regulondb_full


ref_db = load_regulondb_full(drop_duplicates=True)

In [ ]:
BASE_DIR = "/workspace/experiments/11172025_presentation"
RESULTS_DIR = os.path.join(BASE_DIR, "runs_litknowledge1")
FIGURE_DIR = os.path.join(BASE_DIR, "figures_litknowledge")
os.makedirs(FIGURE_DIR, exist_ok=True)

In [ ]:
# directories = glob.glob("/workspace/results/250516_TF_perturbseq/ode_experiment_10282025/*")
directories = glob.glob(f"{RESULTS_DIR}/*")
all_dfs = []
genes_old = None
all_mats = {}
for directory in directories:
    try:
        with open(os.path.join(directory, "config.json"), "r") as f:
            config = json.load(f)

        config_tag = config["tag"]
        mat = np.load(os.path.join(directory, "Amat.npz"), allow_pickle=True)
        a_mat = mat["matrix"]
        genes = mat["genes"]

        if genes_old is None:
            genes_old = genes
        else:
            assert np.all(genes_old == genes), "Genes are not the same"

        all_mats[config_tag] = a_mat
        print(config_tag)
        print(a_mat.shape)

        # print(a_mat[0:5, 0:5])
    except Exception as e:
        print(f"Ignoring {directory}; missing files")

In [ ]:
from ode_script_partiallitknowledge import build_aweight

adata = sc.read_h5ad(config["processing"]["adata_path"])
if config["processing"]["rt_bc"] != "all":
    adata = adata[adata.obs["rt_bc"] == config["processing"]["rt_bc"]].copy()
if config["processing"]["consolidated_cluster"] != "all":
    adata = adata[
        adata.obs["consolidated_cluster"] == config["processing"]["consolidated_cluster"]
    ].copy()
sc.pp.filter_genes(adata, min_cells=10)
aweight = build_aweight(adata, heldout_targets=[], perc_targets_in_training=1.0, random_seed=0)
print(aweight.shape)

In [ ]:
with open("abc_transporter_genes.json", "r") as f:
    heldout_targets = json.load(f)

In [ ]:
a_gt_.flatten().sum()

In [ ]:
mask_ = np.isin(adata.var_names, heldout_targets)
corr_df = []
for config_tag in tqdm(all_mats):
    a_gt_ = (aweight[mask_, :] < 1.0).astype(np.float32)
    a_pred_ = np.abs(all_mats[config_tag][mask_, :])

    ranks = stats.rankdata(-a_pred_.flatten(), method="average")
    true_edge_ranks = ranks[a_gt_.flatten() > 0.5]  # where a_gt_ is True

    print(config_tag)
    print(np.min(true_edge_ranks))

    corr_df.append(
        {"tag": config_tag, "spearmanr": stats.spearmanr(a_gt_.flatten(), a_pred_.flatten())[0]}
    )
corr_df = pd.DataFrame(corr_df)
corr_df.sort_values("spearmanr", ascending=False, inplace=True)
corr_df

In [ ]:
ranks

In [ ]:
plt.hist(all_mats[config_tag].flatten(), bins=100)

In [ ]:
a_pred_.flatten()[63760]

In [ ]:
ranks

In [ ]:
plt.scatter(a_gt_.ravel(), a_pred_.ravel())